# 用 Sciverse 做专利与文献交叉探索

> 同时检索专利和学术文献，发现技术转化与竞争格局

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 检索专利文献

使用 agentic-search 获取相关专利片段


In [ ]:
import httpx

BASE = "https://api.sciverse.space"
TOKEN = "sv-..."
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def search_patents(query: str):
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers=HEADERS,
            json={"query": f"{query} patent", "top_k": 15}
        )
        return resp.json()["hits"]

patent_hits = await search_patents("CRISPR base editing")
print(f"Found {len(patent_hits)} patent-related chunks")

## Step 2: 检索对应学术文献

用 meta-search 按作者、年份筛选学术论文


In [ ]:
async def search_academic(query: str, authors: list = None):
    async with httpx.AsyncClient() as client:
        filters = []
        if authors:
            filters.append({"field": "authors", "op": "in", "value": authors})
        resp = await client.post(
            f"{BASE}/meta-search",
            headers=HEADERS,
            json={"query": query, "filters": filters, "top_k": 20}
        )
        return resp.json()["hits"]

# 检索与专利发明人对应的学术论文
academic_hits = await search_academic(
    "CRISPR base editing adenine cytosine",
    authors=["David Liu", "Nicole Gaudelli"]
)
print(f"Found {len(academic_hits)} academic papers")

## Step 3: 交叉分析与报告生成

将专利与文献关联，生成结构化分析


In [ ]:
from anthropic import Anthropic

client = Anthropic()

patent_summary = "\n".join([
    f"- [{h['doc_id']}] {h['title']}: {h['chunk'][:100]}..."
    for h in patent_hits[:8]
])
academic_summary = "\n".join([
    f"- [{h['doc_id']}] {h['title']} ({h.get('year','')})"
    for h in academic_hits[:8]
])

msg = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=4096,
    messages=[{
        "role": "user",
        "content": f"""分析以下专利与学术文献的关联：

专利：
{patent_summary}

学术论文：
{academic_summary}

请输出：1) 专利持有人分布 2) 专利-论文关联 3) 技术转化路径"""
    }]
)
print(msg.content[0].text)

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
